In [1]:
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver, VQE
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit.circuit.library import EfficientSU2
import numpy as np
from qiskit_algorithms.optimizers import SLSQP , SPSA , ADAM
from qiskit_aer.primitives import Estimator
#qiskit_nature.settings.use_pauli_sum_op = False 
from qiskit_aer import Aer
from pyinstrument import Profiler
import threading
from qiskit_aer.primitives import Estimator as AerEstimator
import math
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import AerSimulator
import cupy as cp
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

In [4]:
molecule = MoleculeInfo(
        # Coordinates in Angstrom
        symbols=["H" , "H"],
        coords=([0.00000, 0.00000, 0.00000], [1.500000, 0.00000, 0.00000]),
        multiplicity=1,  # = 2*spin + 1
        charge=0,
    )

driver = PySCFDriver.from_molecule(molecule)
properties = driver.run()

problem = FreezeCoreTransformer(
     freeze_core=True, 
).transform(properties)

num_particles = problem.num_particles
num_spatial_orbitals = problem.num_spatial_orbitals
print(num_particles)
print(num_spatial_orbitals)

mapper = ParityMapper(num_particles=num_particles)
qubit_op = mapper.map(problem.second_q_ops()[0])
qubit_op

(1, 1)
2


SparsePauliOp(['II', 'IZ', 'ZI', 'ZZ', 'XX'],
              coeffs=[-1.00964469+0.j,  0.12910131+0.j, -0.12910131+0.j, -0.00418896+0.j,
  0.22953594+0.j])

In [5]:
def exact_solver(qubit_op, problem):
    sol = NumPyMinimumEigensolver().compute_minimum_eigenvalue(qubit_op)
    result = problem.interpret(sol)
    return result

result = exact_solver(qubit_op , problem)
print("\n\n------------exact energy------------\n\n" , result)
print(" final ground state energy using exact solver ========> " , result.total_energies[0].real)



------------exact energy------------

 === GROUND STATE ENERGY ===
 
* Electronic ground state energy (Hartree): -1.350934160751
  - computed part:      -1.350934160751
  - FreezeCoreTransformer extracted energy part: 0.0
~ Nuclear repulsion energy (Hartree): 0.35278480728
> Total ground state energy (Hartree): -0.998149353471
 
=== MEASURED OBSERVABLES ===
 
 
=== DIPOLE MOMENTS ===
 
~ Nuclear dipole moment (a.u.): [2.83458919  0.0  0.0]
 
 final ground state energy using exact solver ========>  -0.9981493534714092


In [6]:
init_state = HartreeFock( num_spatial_orbitals, num_particles, mapper)
var_form = UCCSD( num_spatial_orbitals, num_particles, mapper, initial_state=init_state)

In [10]:
# from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2
from qiskit_aer import QasmSimulator , Aer
# backend = QasmSimulator()
# backend = FakeAlmadenV2()
# try:
#     backend = Aer.get_backend('aer_simulator_statevector_gpu')
#     backend.set_options(device='GPU')
#     backend.set_options(cuStateVec_enable=False)
#     print("GPU options set successfully." , backend)
# except Exception as e:
#     print(f"Failed to set GPU options: {e}")

# backend= {
#     "method": "statevector",
#     "precision": "double",
#     "max_parallel_threads": 4,
#     "device": "GPU"
# }

noiseless_estimator = AerEstimator(backend_options = backend , approximation=True)
optimizer = SLSQP(maxiter=10)
aer_estimator = AerEstimator(backend_options=backend , approximation=True)